
# Telecom Multi-Agent Agent-to-Agent Fallback with LangGraph

This notebook demonstrates a **telecom domain multi-agent scenario** where one specialist agent can fall back to another specialist agent using **LangGraph**.

## Scenario

A customer says:

> "My mobile internet is very slow in Pune since morning."

The workflow uses multiple specialist agents:

- **Router Agent**: starts the workflow
- **Network KPI Agent**: checks latency, packet loss, congestion
- **Incident Agent**: checks area outage / maintenance / backhaul issues
- **Billing Agent**: checks plan validity, FUP throttling, recharge status
- **Decision Agent**: decides the most probable cause
- **Response Agent**: produces the final support response
- **Human Review Agent**: fallback when confidence stays low

## Goal

Show how to implement:

- agent-to-agent fallback
- conditional routing
- low-confidence escalation
- fallback history tracking
- optional LangSmith tracing setup



## Why this pattern matters

In telecom, the same user complaint can have many underlying causes:

- network congestion
- site outage
- backhaul degradation
- expired plan
- FUP throttling
- device or SIM issue

A single agent may not be enough.  
A **multi-agent fallback** design lets the workflow try another specialist before failing.


In [ ]:
# %pip install -q langgraph langchain-core

In [ ]:

from typing import TypedDict, Optional, List, Dict, Any
from pprint import pprint


## 1. Mock Telecom Data

In [ ]:

CUSTOMERS = {
    "CUST_1001": {
        "name": "Amit Kulkarni",
        "city": "Pune",
        "area": "Hinjewadi",
        "plan": "5G Unlimited Max",
        "recent_usage_gb": 4.2,
        "sim_status": "active",
        "device_type": "5G Smartphone"
    }
}

NETWORK_KPIS = {
    ("Pune", "Hinjewadi"): {
        "signal_strength_dbm": -109,
        "latency_ms": 190,
        "packet_loss_pct": 8.1,
        "cell_congestion_pct": 92,
        "availability_pct": 98.1
    }
}

INCIDENTS = {
    ("Pune", "Hinjewadi"): {
        "incident_id": "INC-TEL-3007",
        "status": "open",
        "title": "Fiber backhaul degradation impacting sector throughput",
        "severity": "high",
        "eta_hours": 3
    }
}

BILLING = {
    "CUST_1001": {
        "plan_active": True,
        "recharge_status": "success",
        "fup_limit_hit": False,
        "outstanding_amount": 0
    }
}


## 2. Mock Specialist Agent Tool Functions

In [ ]:

def get_customer_profile(customer_id: str) -> Dict[str, Any]:
    if customer_id not in CUSTOMERS:
        raise ValueError(f"Customer {customer_id} not found")
    return CUSTOMERS[customer_id]

def get_network_kpis(city: str, area: str) -> Dict[str, Any]:
    key = (city, area)
    if key not in NETWORK_KPIS:
        raise ValueError(f"No KPI data found for {city}/{area}")
    return NETWORK_KPIS[key]

def get_active_incident(city: str, area: str) -> Optional[Dict[str, Any]]:
    return INCIDENTS.get((city, area))

def get_billing_status(customer_id: str) -> Dict[str, Any]:
    if customer_id not in BILLING:
        raise ValueError(f"No billing data found for {customer_id}")
    return BILLING[customer_id]


## 3. Shared State

In [ ]:

class TelecomState(TypedDict, total=False):
    user_query: str
    customer_id: str
    customer_profile: Dict[str, Any]
    assigned_agent: str
    network_kpis: Dict[str, Any]
    incident_data: Dict[str, Any]
    billing_data: Dict[str, Any]
    probable_cause: str
    recommendation: str
    confidence: float
    error_message: str
    fallback_history: List[str]
    requires_human_review: bool
    final_response: str



## 4. Build the Specialist Agents

Each agent updates the shared state.

The important idea is:

- if one agent fails or has low confidence,
- LangGraph routes to another agent.


In [ ]:

def router_agent(state: TelecomState) -> TelecomState:
    return {
        "customer_id": "CUST_1001",
        "fallback_history": [],
        "assigned_agent": "network_kpi_agent",
        "confidence": 0.0
    }

def customer_agent(state: TelecomState) -> TelecomState:
    profile = get_customer_profile(state["customer_id"])
    return {"customer_profile": profile}

def network_kpi_agent(state: TelecomState) -> TelecomState:
    # Simulate a failure or weak result from the first specialist
    # In real life, this could be:
    # - KPI API timeout
    # - partial data
    # - conflicting measurements
    return {
        "assigned_agent": "network_kpi_agent",
        "error_message": "KPI API timeout while fetching live radio metrics",
        "confidence": 0.30,
        "fallback_history": state.get("fallback_history", []) + ["network_kpi_agent"]
    }

def incident_agent(state: TelecomState) -> TelecomState:
    profile = state["customer_profile"]
    incident = get_active_incident(profile["city"], profile["area"])

    if incident:
        return {
            "assigned_agent": "incident_agent",
            "incident_data": incident,
            "probable_cause": "Active local network issue affecting customer throughput",
            "recommendation": f"Inform customer about known issue and share ETA of {incident['eta_hours']} hours.",
            "confidence": 0.88,
            "error_message": "",
            "fallback_history": state.get("fallback_history", []) + ["incident_agent"]
        }

    return {
        "assigned_agent": "incident_agent",
        "error_message": "No active incident found",
        "confidence": 0.45,
        "fallback_history": state.get("fallback_history", []) + ["incident_agent"]
    }

def billing_agent(state: TelecomState) -> TelecomState:
    billing = get_billing_status(state["customer_id"])

    if billing["plan_active"] and not billing["fup_limit_hit"]:
        return {
            "assigned_agent": "billing_agent",
            "billing_data": billing,
            "probable_cause": state.get("probable_cause", "Billing not likely to be the cause"),
            "recommendation": state.get("recommendation", "Proceed with technical investigation"),
            "confidence": max(state.get("confidence", 0.0), 0.72),
            "error_message": "",
            "fallback_history": state.get("fallback_history", []) + ["billing_agent"]
        }

    if billing["fup_limit_hit"]:
        return {
            "assigned_agent": "billing_agent",
            "billing_data": billing,
            "probable_cause": "Data speed likely reduced due to FUP throttling",
            "recommendation": "Inform customer of FUP status and offer top-up or renewal",
            "confidence": 0.82,
            "error_message": "",
            "fallback_history": state.get("fallback_history", []) + ["billing_agent"]
        }

    return {
        "assigned_agent": "billing_agent",
        "error_message": "Billing validation inconclusive",
        "confidence": 0.40,
        "fallback_history": state.get("fallback_history", []) + ["billing_agent"]
    }

def decision_agent(state: TelecomState) -> TelecomState:
    # Consolidate the best available evidence
    if state.get("incident_data"):
        cause = state.get("probable_cause", "Local telecom incident detected")
        recommendation = state.get("recommendation", "Share ETA and monitor restoration")
        confidence = max(state.get("confidence", 0.0), 0.85)
    elif state.get("billing_data", {}).get("fup_limit_hit"):
        cause = "Customer is likely impacted by FUP throttling"
        recommendation = "Advise customer to top up or wait for cycle refresh"
        confidence = 0.82
    else:
        cause = state.get("probable_cause", "Cause remains uncertain")
        recommendation = state.get("recommendation", "Escalate to telecom support engineer")
        confidence = state.get("confidence", 0.50)

    return {
        "probable_cause": cause,
        "recommendation": recommendation,
        "confidence": confidence
    }

def human_review_agent(state: TelecomState) -> TelecomState:
    return {
        "requires_human_review": True,
        "final_response": (
            "Automated agents could not reach sufficient confidence. "
            "Escalate to telecom operations engineer for manual investigation."
        )
    }

def response_agent(state: TelecomState) -> TelecomState:
    customer = state["customer_profile"]
    incident = state.get("incident_data")

    incident_block = "No active incident found."
    if incident:
        incident_block = (
            f"{incident['incident_id']} | {incident['title']} | "
            f"Severity: {incident['severity']} | ETA: {incident['eta_hours']} hours"
        )

    final = f'''
Telecom Multi-Agent Diagnosis Summary

Customer ID: {state["customer_id"]}
Customer Name: {customer["name"]}
Location: {customer["city"]}, {customer["area"]}
Plan: {customer["plan"]}

Final Probable Cause:
{state.get("probable_cause", "Unknown")}

Recommended Action:
{state.get("recommendation", "Escalate for manual review")}

Confidence:
{state.get("confidence", 0.0):.2f}

Incident Details:
{incident_block}

Fallback Path Used:
{state.get("fallback_history", [])}
'''.strip()

    return {"final_response": final}



## 5. Conditional Routing Logic

This is where **agent-to-agent fallback** happens.

Routing policy:
- Try **Network KPI Agent** first
- If it fails or has low confidence, fallback to **Incident Agent**
- If confidence is still low, fallback to **Billing Agent**
- If confidence is still low, escalate to **Human Review**
- Otherwise continue to **Decision Agent**


In [ ]:

def route_after_network(state: TelecomState) -> str:
    if state.get("error_message") or state.get("confidence", 0) < 0.70:
        return "incident_agent"
    return "decision_agent"

def route_after_incident(state: TelecomState) -> str:
    if state.get("confidence", 0) < 0.70:
        return "billing_agent"
    return "decision_agent"

def route_after_billing(state: TelecomState) -> str:
    if state.get("confidence", 0) < 0.70:
        return "human_review_agent"
    return "decision_agent"


## 6. Build the LangGraph Workflow

In [ ]:

from langgraph.graph import StateGraph, END

graph = StateGraph(TelecomState)

graph.add_node("router_agent", router_agent)
graph.add_node("customer_agent", customer_agent)
graph.add_node("network_kpi_agent", network_kpi_agent)
graph.add_node("incident_agent", incident_agent)
graph.add_node("billing_agent", billing_agent)
graph.add_node("decision_agent", decision_agent)
graph.add_node("human_review_agent", human_review_agent)
graph.add_node("response_agent", response_agent)

graph.set_entry_point("router_agent")
graph.add_edge("router_agent", "customer_agent")
graph.add_edge("customer_agent", "network_kpi_agent")

graph.add_conditional_edges("network_kpi_agent", route_after_network)
graph.add_conditional_edges("incident_agent", route_after_incident)
graph.add_conditional_edges("billing_agent", route_after_billing)

graph.add_edge("decision_agent", "response_agent")
graph.add_edge("response_agent", END)
graph.add_edge("human_review_agent", END)

app = graph.compile()
print("Telecom multi-agent fallback graph compiled successfully.")


## 7. Run the Scenario

In [ ]:

input_state = {
    "user_query": "My mobile internet is very slow in Pune since morning."
}

result = app.invoke(input_state)
print(result["final_response"])



## 8. What happened here?

### Step-by-step
1. Router initialized the workflow
2. Customer profile was loaded
3. Network KPI Agent failed with a KPI timeout
4. LangGraph routed to Incident Agent
5. Incident Agent found an active local network issue
6. Confidence became high enough
7. Decision Agent finalized the diagnosis
8. Response Agent generated the final summary

This is **agent-to-agent fallback** in action.



## 9. Optional Variation: Make Incident Agent fail too

You can modify `incident_agent()` to simulate:
- no outage found
- weak evidence
- low confidence

Then the graph will fall back to `billing_agent`.



## 10. Optional LangSmith Tracing Setup

Uncomment and set these values if you want LangSmith traces.

This helps you inspect:
- which agent failed
- which fallback route was used
- final confidence
- latency and execution path


In [ ]:

# import os
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = "your_langsmith_api_key"
# os.environ["LANGCHAIN_PROJECT"] = "telecom-multi-agent-fallback"



## 11. Practice Exercises

Try these extensions:

1. Make `network_kpi_agent()` succeed and see how fallback changes
2. Make `incident_agent()` return low confidence so billing fallback is used
3. Add a **validator agent**
4. Add a **RAG/SOP agent**
5. Add a **human approval node** for high-risk decisions
6. Add retry counters and failure count in state



## 12. Final Takeaway

This notebook shows a practical **telecom domain multi-agent fallback pattern** using LangGraph.

### Key idea
Do not let one weak specialist break the workflow.

Instead:
- detect failure or low confidence
- route to the next best specialist
- preserve context in shared state
- escalate only when needed

That is how to design resilient multi-agent telecom workflows.
